In [1]:
import pandas as pd
import numpy as np
import scipy.io as sio
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score, confusion_matrix
from datetime import datetime, timedelta
import warnings

# 忽略一些 pandas 的警告
warnings.filterwarnings('ignore')

In [2]:
# 檔案路徑 (請根據你的環境修改)
PATH_DATASET = '../dataset/NEUSTG_19502020_12stations.mat'
PATH_THRESHOLDS = '../dataset/Seed_Coastal_Stations_Thresholds.mat'
# PATH_INTERVALS = '../dataset/Seed_Historical_Time_Intervals.txt' # 假設在當前目錄
HIST_WINDOW = 7   # 使用過去 7 天
PRED_WINDOW = 14  # 預測未來 14 天

In [3]:
# ============================================
# 1. 競賽設定 (依據任務說明)
# ============================================
TRAINING_STATIONS = ['Annapolis','Atlantic_City','Charleston','Washington','Wilmington', 
                     'Eastport', 'Portland', 'Sewells_Point', 'Sandy_Hook']
TESTING_STATIONS = ['Lewes', 'Fernandina_Beach', 'The_Battery']
# all_station_names = TRAINING_STATIONS+TESTING_STATIONS

In [4]:
def matlab2datetime(matlab_datenum):
    """將 MATLAB datenum 轉換為 Python datetime"""
    day = datetime.fromordinal(int(matlab_datenum))
    dayfrac = timedelta(days=matlab_datenum % 1) - timedelta(days=366)
    return day + dayfrac

def parse_mat_strings(mat_array):
    """穩健地解析 MATLAB 的字串陣列"""
    # 展平陣列確保遍歷所有元素
    flat_array = mat_array.flatten()
    parsed_strings = []
    for item in flat_array:
        # 處理可能的嵌套 array(['Name'])
        if isinstance(item, np.ndarray) and item.size > 0:
            parsed_strings.append(str(item.item())) # .item() 取出純量
        # 處理直接是字串的情況
        elif isinstance(item, str):
            parsed_strings.append(item)
        # 處理其他可能的嵌套結構
        else:
            try:
                parsed_strings.append(str(item[0]))
            except:
                parsed_strings.append(str(item))
    return parsed_strings

def load_data(path_data, path_thresholds):
    """讀取並合併數據與閾值 (包含所有站點)"""
    print(f"正在讀取資料: {path_data}")
    data = sio.loadmat(path_data)
    
    lat = data['lattg'].flatten()
    lon = data['lontg'].flatten()
    time = data['t'].flatten()
    sea_level = data['sltg'] # Shape: (Time, Stations)
    
    # 解析站點名稱 (關鍵修正)
    station_names = parse_mat_strings(data['sname'])
    print(f"   讀取到的站點列表 ({len(station_names)}個): {station_names}")
    
    # 轉換時間
    time_dt = [matlab2datetime(t) for t in time]
    
    # 建立 DataFrame
    records = []
    for i, name in enumerate(station_names):
        df_temp = pd.DataFrame({
            'time': time_dt,
            'station_name': name,
            'latitude': lat[i],
            'longitude': lon[i],
            'sea_level': sea_level[:, i]
        })
        records.append(df_temp)
    
    df_hourly = pd.concat(records, ignore_index=True)
    df_hourly['time'] = pd.to_datetime(df_hourly['time'])
    
    # 讀取並合併官方閾值
    thresh_data = sio.loadmat(path_thresholds)
    th_names = parse_mat_strings(thresh_data['sname'])
    th_vals = thresh_data['thminor_stnd'].flatten()
    
    threshold_map = dict(zip(th_names, th_vals))
    
    # Map 到主數據
    df_hourly['flood_threshold'] = df_hourly['station_name'].map(threshold_map)
    df_hourly['is_flood'] = (df_hourly['sea_level'] > df_hourly['flood_threshold']).astype(int)
    
    return df_hourly

In [5]:
# 3. 特徵工程
# ============================================
def preprocess_daily(df_hourly):
    """將小時數據轉為日數據並加入特徵"""
    print("正在進行每日聚合與特徵工程...")
    # 轉為每日：計算日均水位，並判斷當天是否發生過淹水
    df_daily = df_hourly.groupby(['station_name', pd.Grouper(key='time', freq='D')]).agg({
        'sea_level': 'mean',
        'is_flood': 'max',
        'latitude': 'first',
        'longitude': 'first'
    }).reset_index()
    
    # === 特徵工程 ===
    # 1. 季節性
    df_daily['month_sin'] = np.sin(2 * np.pi * df_daily['time'].dt.month / 12)
    df_daily['month_cos'] = np.cos(2 * np.pi * df_daily['time'].dt.month / 12)
    
    # 2. 趨勢 (年份正規化)
    df_daily['year_norm'] = (df_daily['time'].dt.year - 1950) / 70
    
    # 3. 滑動視窗統計
    for window in [3, 7]:
        grp = df_daily.groupby('station_name')['sea_level']
        df_daily[f'sl_mean_{window}d'] = grp.transform(lambda x: x.rolling(window, min_periods=1).mean())
        df_daily[f'sl_max_{window}d'] = grp.transform(lambda x: x.rolling(window, min_periods=1).max())
        df_daily[f'sl_std_{window}d'] = grp.transform(lambda x: x.rolling(window, min_periods=1).std())
            
    return df_daily.fillna(0)

In [6]:
# 4. 構建訓練數據 (序列化)
# ============================================
def create_sequences(df, stations, hist_days=7, future_days=14, feature_cols=None):
    X, y = [], []
    df_subset = df[df['station_name'].isin(stations)].copy()
    
    for stn, group in df_subset.groupby('station_name'):
        group = group.sort_values('time').reset_index(drop=True)
        vals = group[feature_cols].values
        targets = group['is_flood'].values
        
        # 這裡簡單使用步長為 1 的滑動窗口
        # 實際比賽中應注意避免 Validation 資料洩漏
        for i in range(0, len(group) - hist_days - future_days, 1):
            x_seq = vals[i : i + hist_days].flatten()
            y_seq = targets[i + hist_days : i + hist_days + future_days]
            X.append(x_seq)
            y.append(y_seq)
            
    return np.array(X), np.array(y)

In [7]:
# 1. 載入資料
df_hourly = load_data(PATH_DATASET, PATH_THRESHOLDS)
df_daily = preprocess_daily(df_hourly)
df_daily.head()

正在讀取資料: ../dataset/NEUSTG_19502020_12stations.mat
   讀取到的站點列表 (12個): ['Annapolis', 'Atlantic_City', 'Charleston', 'Eastport', 'Fernandina_Beach', 'Lewes', 'Portland', 'Sandy_Hook', 'Sewells_Point', 'The_Battery', 'Washington', 'Wilmington']
正在進行每日聚合與特徵工程...


,station_name,time,sea_level,is_flood,latitude,longitude,month_sin,month_cos,year_norm,sl_mean_3d,sl_max_3d,sl_std_3d,sl_mean_7d,sl_max_7d,sl_std_7d
0,Annapolis,1950-01-01,1.432625,0,38.98328,-76.4816,0.5,0.866025,0.0,1.432625,1.432625,0.000000,1.432625,1.432625,0.000000
1,Annapolis,1950-01-02,1.460625,0,38.98328,-76.4816,0.5,0.866025,0.0,1.446625,1.460625,0.019799,1.446625,1.460625,0.019799
2,Annapolis,1950-01-03,1.486042,0,38.98328,-76.4816,0.5,0.866025,0.0,1.459764,1.486042,0.026719,1.459764,1.486042,0.026719
3,Annapolis,1950-01-04,1.511333,0,38.98328,-76.4816,0.5,0.866025,0.0,1.486000,1.511333,0.025354,1.472656,1.511333,0.033775
4,Annapolis,1950-01-05,1.281542,0,38.98328,-76.4816,0.5,0.866025,0.0,1.426306,1.511333,0.126005,1.434433,1.511333,0.090336


In [8]:
# 2. 準備訓練數據
FEAT_COLS = ['sea_level', 'month_sin', 'month_cos', 'year_norm', 
             'sl_mean_3d', 'sl_max_3d', 'sl_mean_7d', 'sl_max_7d', 'sl_std_7d']

print(f"\n正在準備訓練資料 (使用 {len(TRAINING_STATIONS)} 個訓練站點)...")
X_train, y_train = create_sequences(df_daily, TRAINING_STATIONS, HIST_WINDOW, PRED_WINDOW, FEAT_COLS)


正在準備訓練資料 (使用 9 個訓練站點)...


In [9]:
# 計算權重 (處理類別不平衡)
pos_count = np.sum(y_train == 1)
neg_count = np.sum(y_train == 0)
scale_weight = neg_count / pos_count if pos_count > 0 else 1
print(f"   資料形狀: X={X_train.shape}, y={y_train.shape}")
print(f"   正樣本: {pos_count}, 負樣本: {neg_count}, 建議 scale_pos_weight: {scale_weight:.2f}")

   資料形狀: X=(233208, 63), y=(233208, 14)
   正樣本: 113454, 負樣本: 3151458, 建議 scale_pos_weight: 27.78


In [10]:
# 3. 訓練模型
print("\n開始訓練 14 個 XGBoost 模型...")
models = []
for d in range(PRED_WINDOW):
    print(f"   Training Day {d+1}...", end='\r')
    clf = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        scale_pos_weight=scale_weight,
        eval_metric='logloss',
        use_label_encoder=False,
        n_jobs=-1,
        random_state=42
    )
    clf.fit(X_train, y_train[:, d])
    models.append(clf)
print("\n訓練完成！")


開始訓練 14 個 XGBoost 模型...
   Training Day 14...
訓練完成！


In [11]:
# 5. 官方區間評估 (Hardcoded Intervals)
# ============================================
print("\n=== 開始官方區間評估 ===")

# 直接定義測試區間 (避免讀檔錯誤)
intervals_data = {
    'start_date': [
        '3/6/1962', '7/21/2013', '5/13/2011', '12/21/1995', '9/5/1995',
        '12/31/2009', '9/16/2020', '10/7/2013', '4/3/1958', '5/13/2011',
        '4/8/1988', '12/4/1996', '4/14/2003', '1/25/1979', '3/18/2015'
    ],
    'end_date': [
        '3/12/1962', '7/27/2013', '5/19/2011', '12/27/1995', '9/11/1995',
        '1/6/2010', '9/22/2020', '10/13/2013', '4/9/1958', '5/19/2011',
        '4/14/1988', '12/10/1996', '4/20/2003', '1/31/1979', '3/24/2015'
    ]
}
test_intervals = pd.DataFrame(intervals_data)

all_f1, all_mcc, all_acc = [], [], []

for idx, row in test_intervals.iterrows():
    h_start = pd.to_datetime(row['start_date'])
    h_end = pd.to_datetime(row['end_date'])
    pred_start = h_end + timedelta(days=1)
    pred_end = pred_start + timedelta(days=13)
    
    y_true_interval = []
    y_pred_interval = []
    
    # 針對三個測試站點進行預測
    for stn in TESTING_STATIONS:
        stn_df = df_daily[df_daily['station_name'] == stn]
        
        # 提取資料
        mask_hist = (stn_df['time'] >= h_start) & (stn_df['time'] <= h_end)
        mask_future = (stn_df['time'] >= pred_start) & (stn_df['time'] <= pred_end)
        
        hist_data = stn_df.loc[mask_hist, FEAT_COLS].values.flatten()
        true_future = stn_df.loc[mask_future, 'is_flood'].values
        
        # 檢查完整性
        expected_hist_len = HIST_WINDOW * len(FEAT_COLS)
        if len(hist_data) == expected_hist_len and len(true_future) == PRED_WINDOW:
            # 預測
            preds = []
            for model in models:
                preds.append(model.predict([hist_data])[0])
            
            y_pred_interval.extend(preds)
            y_true_interval.extend(true_future)
    
    # 計算指標
    if len(y_true_interval) > 0:
        f1 = f1_score(y_true_interval, y_pred_interval, zero_division=0)
        mcc = matthews_corrcoef(y_true_interval, y_pred_interval)
        acc = accuracy_score(y_true_interval, y_pred_interval)
        all_f1.append(f1)
        all_mcc.append(mcc)
        all_acc.append(acc)
        print(f"   區間 {idx+1} ({h_start.date()}): F1={f1:.3f}, MCC={mcc:.3f}")
    else:
        print(f"   區間 {idx+1}: 無有效資料 (可能缺值)")

print("\n" + "="*30)
if all_f1:
    print(f"最終結果 (共 {len(all_f1)} 個有效區間):")
    print(f"平均 F1 Score: {np.mean(all_f1):.4f}")
    print(f"平均 MCC     : {np.mean(all_mcc):.4f}")
    print(f"平均 Accuracy: {np.mean(all_acc):.4f}")
else:
    print("無法計算分數，請檢查資料載入是否正確。")
print("="*30)


=== 開始官方區間評估 ===
   區間 1 (1962-03-06): F1=0.000, MCC=0.000
   區間 2 (2013-07-21): F1=0.000, MCC=0.000
   區間 3 (2011-05-13): F1=0.000, MCC=0.000
   區間 4 (1995-12-21): F1=0.000, MCC=-0.125
   區間 5 (1995-09-05): F1=0.000, MCC=0.000
   區間 6 (2009-12-31): F1=0.000, MCC=0.000
   區間 7 (2020-09-16): F1=0.047, MCC=0.000
   區間 8 (2013-10-07): F1=0.000, MCC=0.000
   區間 9 (1958-04-03): F1=0.000, MCC=0.000
   區間 10 (2011-05-13): F1=0.000, MCC=0.000
   區間 11 (1988-04-08): F1=0.000, MCC=0.000
   區間 12 (1996-12-04): F1=0.296, MCC=0.295
   區間 13 (2003-04-14): F1=0.000, MCC=0.000
   區間 14 (1979-01-25): F1=0.000, MCC=0.000
   區間 15 (2015-03-18): F1=0.000, MCC=0.000

最終結果 (共 15 個有效區間):
平均 F1 Score: 0.0229
平均 MCC     : 0.0113
平均 Accuracy: 0.4222


In [12]:
# ============================================
# 5. 進行官方測試區間評估 (Debug 版)
# ============================================
print("\n=== 開始 Debug 評估流程 ===")

# A. 手動建立測試區間 (避免檔案讀取錯誤)
# 直接將你提供的測試區間寫入程式碼，確保資料存在
intervals_data = {
    'start_date': [
        '3/6/1962', '7/21/2013', '5/13/2011', '12/21/1995', '9/5/1995',
        '12/31/2009', '9/16/2020', '10/7/2013', '4/3/1958', '5/13/2011',
        '4/8/1988', '12/4/1996', '4/14/2003', '1/25/1979', '3/18/2015'
    ],
    'end_date': [
        '3/12/1962', '7/27/2013', '5/19/2011', '12/27/1995', '9/11/1995',
        '1/6/2010', '9/22/2020', '10/13/2013', '4/9/1958', '5/19/2011',
        '4/14/1988', '12/10/1996', '4/20/2003', '1/31/1979', '3/24/2015'
    ]
}
test_intervals = pd.DataFrame(intervals_data)
print(f"1. 測試區間載入成功: 共 {len(test_intervals)} 筆")

# B. 檢查站點名稱匹配
print("2. 檢查站點名稱:")
print(f"   我們設定的測試站點: {TESTING_STATIONS}")
actual_stations = df_daily['station_name'].unique()
print(f"   資料集中的站點範例: {actual_stations[:3]}")

# 檢查是否有測試站點不在資料集中
missing_stations = [s for s in TESTING_STATIONS if s not in actual_stations]
if missing_stations:
    print(f"   [警告] 找不到這些測試站點: {missing_stations} (請檢查大小寫或空白鍵)")
else:
    print("   [OK] 所有測試站點名稱匹配成功")

# C. 執行評估迴圈
all_f1, all_mcc, all_acc = [], [], []

for idx, row in test_intervals.iterrows():
    # 解析日期
    h_start = pd.to_datetime(row['start_date'])
    h_end = pd.to_datetime(row['end_date'])
    
    # 預測窗口 (歷史結束的隔天開始 14 天)
    pred_start = h_end + timedelta(days=1)
    pred_end = pred_start + timedelta(days=13)
    
    y_true_interval = []
    y_pred_interval = []
    
    # Debug: 印出第一個區間的日期範圍
    if idx == 0:
        print(f"   [範例] 區間 1: 歷史 {h_start.date()}~{h_end.date()} -> 預測 {pred_start.date()}~{pred_end.date()}")
    
    for stn in TESTING_STATIONS:
        stn_df = df_daily[df_daily['station_name'] == stn]
        
        # 提取歷史特徵 (7 days)
        mask_hist = (stn_df['time'] >= h_start) & (stn_df['time'] <= h_end)
        hist_data = stn_df.loc[mask_hist, FEAT_COLS].values.flatten()
        
        # 提取真實標籤 (14 days)
        mask_future = (stn_df['time'] >= pred_start) & (stn_df['time'] <= pred_end)
        true_future = stn_df.loc[mask_future, 'is_flood'].values
        
        # 嚴格檢查資料完整性
        expected_hist_len = HIST_WINDOW * len(FEAT_COLS)
        
        if len(hist_data) == expected_hist_len and len(true_future) == PRED_WINDOW:
            # 進行預測
            for i, model in enumerate(models):
                p = model.predict([hist_data])[0]
                y_pred_interval.append(p)
            y_true_interval.extend(true_future)
        else:
            # 只在第一個區間報錯，避免洗版
            if idx == 0:
                print(f"      [跳過] {stn}: 資料長度不足 (歷史: {len(hist_data)}/{expected_hist_len}, 未來: {len(true_future)}/{PRED_WINDOW})")
                # 可能是因為該日期區間該站點剛好缺資料
    
    # 計算該 Interval 的指標
    if len(y_true_interval) > 0:
        f1 = f1_score(y_true_interval, y_pred_interval, zero_division=0)
        mcc = matthews_corrcoef(y_true_interval, y_pred_interval)
        acc = accuracy_score(y_true_interval, y_pred_interval)
        
        all_f1.append(f1)
        all_mcc.append(mcc)
        all_acc.append(acc)
    else:
        if idx == 0:
            print("   [警告] 區間 1 沒有收集到任何有效預測結果。")

print("\n" + "="*30)
if len(all_f1) > 0:
    print(f"最終結果 (共 {len(all_f1)} 個區間):")
    print(f"平均 F1 Score: {np.mean(all_f1):.4f}")
    print(f"平均 MCC     : {np.mean(all_mcc):.4f}")
    print(f"平均 Accuracy: {np.mean(all_acc):.4f}")
else:
    print("無法計算分數：沒有成功預測任何區間。")
    print("可能原因：1. 日期格式解析錯誤 2. 測試站點在那段時間缺資料 3. 站點名稱不匹配")
print("="*30)


=== 開始 Debug 評估流程 ===
1. 測試區間載入成功: 共 15 筆
2. 檢查站點名稱:
   我們設定的測試站點: ['Lewes', 'Fernandina_Beach', 'The_Battery']
   資料集中的站點範例: ['Annapolis' 'Atlantic_City' 'Charleston']
   [OK] 所有測試站點名稱匹配成功
   [範例] 區間 1: 歷史 1962-03-06~1962-03-12 -> 預測 1962-03-13~1962-03-26

最終結果 (共 15 個區間):
平均 F1 Score: 0.0229
平均 MCC     : 0.0113
平均 Accuracy: 0.4222


In [13]:
import pickle

# 將訓練好的 14 個模型打包儲存
with open('model.pkl', 'wb') as f:
    pickle.dump(models, f)

print("model.pkl 已成功儲存！")

model.pkl 已成功儲存！
